In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from scipy import stats
nltk.download('vader_lexicon', quiet=True)

True

In [7]:
news_df=pd.read_csv('../data/raw/raw_analyst_ratings.csv')


In [ ]:
#sentiment analysis using textblob
def analyze_with_textblob(sentences):
    results = []
    for s in sentences:
        blob = TextBlob(s)
        polarity = blob.sentiment.polarity
        subjectivity = blob.sentiment.subjectivity
        label = "positive" if polarity > 0.1 else "negative" if polarity < -0.1 else "neutral"
        results.append({
            "sentence": s,
            "polarity": polarity,
            "subjectivity": subjectivity,
            "textblob_label": label
        })
    return pd.DataFrame(results)

tb_df = analyze_with_textblob(news_df['headline'])
tb_df

In [8]:
#date alignment
news_df.head()
# news_df['stock']
# news_df['date']=news_df['date'].normalize()
# news_df['date'].dt.dz
# news_df['datetime'] = pd.to_datetime(news_df['date'], format="%Y-%m-%d %H:%M:%S")
# failed_rows = pd.to_datetime(news_df['date'], format="%Y-%m-%d %H:%M:%S", errors='coerce').isna()
# print(news_df[failed_rows]['date'].unique())

news_df['datetime'] = pd.to_datetime(news_df['date'], format='mixed', utc=True)
news_df['datetime']
news_df['date_normalized'] = news_df['datetime'].dt.tz_localize(None).dt.date
# news_df['date_normalized']


In [6]:
pip install pandas pandas-market-calendars

Note: you may need to restart the kernel to use updated packages.


In [9]:
import pandas as pd
from pandas_market_calendars import get_calendar

# Load NYSE calendar
nyse = get_calendar("NYSE")

# Preload full trading calendar ONCE
full_schedule = nyse.schedule(
    start_date="2010-01-01",
    end_date="2030-12-31"
)

trading_days = full_schedule.index.date


def align_to_trading_day(ts):

    # Handle missing values
    if pd.isna(ts):
        return pd.NaT

    # Ensure datetime
    ts = pd.to_datetime(ts, utc=True, errors="coerce")

    if pd.isna(ts):
        return pd.NaT

    # Convert timezone
    ts = ts.tz_convert("America/New_York")

    current_date = ts.date()

    # Find next trading day index
    next_idx = trading_days.searchsorted(current_date)

    if next_idx >= len(trading_days):
        return pd.NaT

    trading_day = trading_days[next_idx]

    # Weekend or holiday → next trading day
    if trading_day != current_date:
        return trading_day

    # Market close check
    market_close = full_schedule.iloc[next_idx]["market_close"]

    # After-hours move to next trading day
    if ts > market_close:
        if next_idx + 1 < len(trading_days):
            return trading_days[next_idx + 1]

    return trading_day


# Use MAP (recommended)
news_df["trading_day"] = news_df["datetime"].map(align_to_trading_day)

In [10]:
# news_df['date_normalized']
apple_df=pd.read_csv('../data/raw/AAPL.csv')
amazon_df=pd.read_csv('../data/raw/AMZN.csv')
google_df=pd.read_csv('../data/raw/GOOG.csv')
meta_df=pd.read_csv('../data/raw/META.csv')
nvda_df=pd.read_csv('../data/raw/NVDA.csv')
apple_df["Date"] = pd.to_datetime(apple_df["Date"]).dt.date
amazon_df["Date"] = pd.to_datetime(amazon_df["Date"]).dt.date
google_df["Date"] = pd.to_datetime(google_df["Date"]).dt.date
meta_df["Date"] = pd.to_datetime(meta_df["Date"]).dt.date
nvda_df["Date"] = pd.to_datetime(nvda_df["Date"]).dt.date

In [11]:
apple_df["return"] = apple_df["Close"].pct_change() * 100
amazon_df["return"] = amazon_df["Close"].pct_change() * 100
google_df["return"] = google_df["Close"].pct_change() * 100
meta_df["return"] = meta_df["Close"].pct_change() * 100
nvda_df["return"] = nvda_df["Close"].pct_change() * 100

In [12]:
# apple_df['return']
stocks_df = pd.concat([
    apple_df,
    amazon_df,
    google_df,
    meta_df,
    nvda_df
], ignore_index=True)
stocks_df.head()

,Date,Close,High,Low,Open,Volume,return
0,2009-01-02,2.721686,2.730385,2.554037,2.575630,746015200,NaN
1,2009-01-05,2.836553,2.884539,2.780469,2.794266,1181608400,4.220416
2,2009-01-06,2.789767,2.914229,2.770872,2.877641,1289310400,-1.649399
3,2009-01-07,2.729484,2.774170,2.706990,2.753477,753048800,-2.160860
4,2009-01-08,2.780169,2.793666,2.700393,2.712090,673500800,1.856959


In [13]:
stocks_df["Date"] = pd.to_datetime(stocks_df["Date"]).dt.date

news_df["trading_day"] = pd.to_datetime(
    news_df["trading_day"]
).dt.date

# Merge datasets
combined = pd.merge(
    news_df,
    stocks_df,
    left_on="trading_day",
    right_on="Date",
    how="left"
)

print(combined.head())

   Unnamed: 0                                 headline  \
0           0  Stocks That Hit 52-Week Highs On Friday   
1           0  Stocks That Hit 52-Week Highs On Friday   
2           0  Stocks That Hit 52-Week Highs On Friday   
3           0  Stocks That Hit 52-Week Highs On Friday   
4           0  Stocks That Hit 52-Week Highs On Friday   

                                                 url          publisher  \
0  https://www.benzinga.com/news/20/06/16190091/s...  Benzinga Insights   
1  https://www.benzinga.com/news/20/06/16190091/s...  Benzinga Insights   
2  https://www.benzinga.com/news/20/06/16190091/s...  Benzinga Insights   
3  https://www.benzinga.com/news/20/06/16190091/s...  Benzinga Insights   
4  https://www.benzinga.com/news/20/06/16190091/s...  Benzinga Insights   

                        date stock                  datetime date_normalized  \
0  2020-06-05 10:30:54-04:00     A 2020-06-05 14:30:54+00:00      2020-06-05   
1  2020-06-05 10:30:54-04:00     A 2020-

In [ ]:


daily_sentiment = (
    news_df
    .groupby(["trading_day", "stock"])["sentiment"]
    .mean()
    .reset_index()
)

# Rename column for clarity
daily_sentiment = daily_sentiment.rename(
    columns={"sentiment": "avg_sentiment"}
)
# Compute stock returns

stocks_df["return"] = (
    stocks_df
    .groupby("stock")["Close"]
    .pct_change() * 100
)

#  Match date formats

stocks_df["Date"] = pd.to_datetime(
    stocks_df["Date"]
).dt.date

daily_sentiment["trading_day"] = pd.to_datetime(
    daily_sentiment["trading_day"]
).dt.date

#  Merge sentiment with returns

combined = pd.merge(
    daily_sentiment,
    stocks_df,
    left_on=["trading_day", "stock"],
    right_on=["Date", "stock"],
    how="left"
)

# View results

print(combined.head())

In [ ]:
#pearson

#  COMPUTE DAILY STOCK RETURNS

# Compute percentage returns separately for each stock
stocks_df["return"] = (
    stocks_df
    .groupby("stock")["Close"]
    .pct_change() * 100
)

#  ENSURE DATE FORMATS MATCH

# Stock dates
stocks_df["Date"] = pd.to_datetime(
    stocks_df["Date"]
).dt.date

# News trading days
news_df["trading_day"] = pd.to_datetime(
    news_df["trading_day"]
).dt.date

#  COMPUTE AVERAGE DAILY SENTIMENT

daily_sentiment = (
    news_df
    .groupby(["trading_day", "stock"])["sentiment"]
    .mean()
    .reset_index()
)

# Rename for clarity
daily_sentiment = daily_sentiment.rename(
    columns={"sentiment": "avg_sentiment"}
)

#  MERGE SENTIMENT WITH STOCK RETURNS

combined = pd.merge(
    daily_sentiment,
    stocks_df,
    left_on=["trading_day", "stock"],
    right_on=["Date", "stock"],
    how="left"
)

# ---------------------------------------------------
# STEP 5: REMOVE MISSING VALUES
# ---------------------------------------------------

combined = combined.dropna(
    subset=["avg_sentiment", "return"]
)

# ---------------------------------------------------
# STEP 6: CALCULATE PEARSON CORRELATION
# ---------------------------------------------------

correlation = combined["avg_sentiment"].corr(
    combined["return"],
    method="pearson"
)

print("\nPearson Correlation:")
print(correlation)

# ---------------------------------------------------
# STEP 7: DISPLAY SAMPLE DATA
# ---------------------------------------------------

print("\nCombined Dataset Sample:")
print(
    combined[
        [
            "trading_day",
            "stock",
            "avg_sentiment",
            "return"
        ]
    ].head()
)

# ---------------------------------------------------
# STEP 8: VISUALIZE RELATIONSHIP
# ---------------------------------------------------

plt.figure(figsize=(8, 6))

plt.scatter(
    combined["avg_sentiment"],
    combined["return"]
)

plt.xlabel("Average Daily Sentiment")
plt.ylabel("Daily Stock Return (%)")
plt.title("Sentiment vs Stock Returns")

plt.grid(True)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Calculate correlation
corr = combined["avg_sentiment"].corr(combined["return"])

# Create scatter plot
plt.figure(figsize=(8, 6))

plt.scatter(
    combined["avg_sentiment"],
    combined["return"]
)

plt.xlabel("Average Daily Sentiment")
plt.ylabel("Daily Stock Return (%)")
plt.title("Sentiment vs Stock Returns")

# Annotate correlation value on the plot
plt.text(
    0.05, 0.95,
    f"Pearson r = {corr:.2f}",
    transform=plt.gca().transAxes,
    fontsize=12,
    verticalalignment='top'
)

plt.grid(True)
plt.show()

In [ ]:
def classify_sentiment(score):
    if score > 0.1:
        return "Positive"
    elif score < -0.1:
        return "Negative"
    else:
        return "Neutral"

combined["sentiment_class"] = combined["avg_sentiment"].apply(classify_sentiment)

avg_returns = combined.groupby("sentiment_class")["return"].mean().reset_index()
#bar chart

plt.figure(figsize=(8, 6))

plt.bar(
    avg_returns["sentiment_class"],
    avg_returns["return"]
)

plt.xlabel("Sentiment Category")
plt.ylabel("Average Daily Return (%)")
plt.title("Average Stock Return by Sentiment Category")

plt.grid(axis="y")
plt.show()